In [ ]:
# ============================
# ATM PROJECT - FULL PYTHON SCRIPT (GOOGLE COLAB)
# ============================

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)

# ---------- 1. LOAD DATA ----------
file_name = "DAF_PROJECT.xlsx"   # make sure this is uploaded in Colab (left panel -> Files)
df = pd.read_excel(file_name, sheet_name=0)

print("Initial shape:", df.shape)
print("Initial columns:", df.columns.tolist())

# Strip spaces from column names
df.columns = [c.strip() for c in df.columns]

# ---------- 2. CONVERT DATA TYPES ----------

# Date columns
for col in ["Date", "Last_Refill_Date", "transaction_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Numeric columns (only convert if they exist)
numeric_cols = [
    "Withdrawals_Count", "Cash_Withdrawn", "Cash_Refilled", "Cash_Remaining",
    "Max_Cash_Capacity", "withdrawal_amount", "refill_amount",
    "atm_balance_after_transaction", "avg_daily_withdrawal",
    "days_since_last_refill", "balance_ratio", "refill_needed_flag",
    "suggested_refill_amount"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nDtypes after conversion:")
print(df.dtypes)

# ---------- 3. BASIC CLEANING ----------

# Remove repeated header-like rows where ATM_ID has header text
if "ATM_ID" in df.columns:
    df = df[~df["ATM_ID"].astype(str).str.upper().isin(["ATM_ID", "ATM ID", "ATM"])]

# Drop rows with missing ATM_ID or Date
if "ATM_ID" in df.columns and "Date" in df.columns:
    df = df.dropna(subset=["ATM_ID", "Date"])

# Drop exact duplicate rows
df = df.drop_duplicates()

print("\nShape after removing header-rows, missing keys & duplicates:", df.shape)

# ---------- 4. HANDLE MISSING VALUES ----------

# Fill numeric NaNs for key metrics
for col in ["Withdrawals_Count", "Cash_Withdrawn", "Cash_Refilled", "Cash_Remaining"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Fill text NaNs
for col in ["Branch", "location_city"]:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

# ---------- 5. CREATE DAY_OF_WEEK & HOLIDAY_FLAG ----------

if "Date" in df.columns:
    df["Day_of_Week"] = df["Date"].dt.day_name()
    df["Holiday_Flag"] = df["Day_of_Week"].isin(["Saturday", "Sunday"]).astype(int)
else:
    print("WARNING: 'Date' column not found. Cannot compute Day_of_Week or Holiday_Flag")

# ---------- 6. SORT DATA ----------

sort_cols = []
if "ATM_ID" in df.columns:
    sort_cols.append("ATM_ID")
if "Date" in df.columns:
    sort_cols.append("Date")

if sort_cols:
    df = df.sort_values(sort_cols).reset_index(drop=True)

# ---------- 7. MAX CAPACITY FILL (if needed) ----------

if "Max_Cash_Capacity" in df.columns and "ATM_ID" in df.columns:
    df["Max_Cash_Capacity"] = df.groupby("ATM_ID")["Max_Cash_Capacity"].transform(
        lambda x: x.fillna(x.max())
    )

# ---------- 8. LAST REFILL & DAYS_SINCE_LAST_REFILL ----------

if all(col in df.columns for col in ["ATM_ID", "Date", "Cash_Refilled"]):
    # Last_Refill_Date: last date when Cash_Refilled > 0, forward-filled
    df["Last_Refill_Date"] = df.groupby("ATM_ID").apply(
        lambda g: g["Date"].where(g["Cash_Refilled"] > 0).ffill()
    ).reset_index(level=0, drop=True)

    # days_since_last_refill: difference between Date and Last_Refill_Date
    df["days_since_last_refill"] = (df["Date"] - df["Last_Refill_Date"]).dt.days
    df["days_since_last_refill"] = df["days_since_last_refill"].fillna(0).astype(int)
else:
    print("WARNING: Cannot compute Last_Refill_Date and days_since_last_refill (missing one of ATM_ID, Date, Cash_Refilled).")

# ---------- 9. BALANCE RATIO & REFILL LOGIC ----------

if "Cash_Remaining" in df.columns and "Max_Cash_Capacity" in df.columns:
    df["balance_ratio"] = df["Cash_Remaining"] / df["Max_Cash_Capacity"]
    df["balance_ratio"] = df["balance_ratio"].fillna(0).clip(0, 1)
else:
    print("WARNING: Cannot compute balance_ratio (missing Cash_Remaining or Max_Cash_Capacity).")

# Simple refill rule: refill_needed_flag
if "ATM_ID" in df.columns and "days_since_last_refill" in df.columns and "balance_ratio" in df.columns:
    median_days = df.groupby("ATM_ID")["days_since_last_refill"].transform("median").fillna(3)
    df["refill_needed_flag"] = ((df["balance_ratio"] < 0.25) | (df["days_since_last_refill"] > median_days)).astype(int)
else:
    print("WARNING: Cannot compute refill_needed_flag (missing some required columns).")

# suggested_refill_amount: top up to 90% capacity
if "Max_Cash_Capacity" in df.columns and "Cash_Remaining" in df.columns:
    df["suggested_refill_amount"] = ((df["Max_Cash_Capacity"] * 0.9) - df["Cash_Remaining"]).clip(lower=0)
else:
    print("WARNING: Cannot compute suggested_refill_amount (missing Max_Cash_Capacity or Cash_Remaining).")

# ---------- 10. BASIC SUMMARY FOR REPORT ----------

print("\nBASIC SUMMARY")
if "ATM_ID" in df.columns:
    print("Number of ATMs:", df["ATM_ID"].nunique())
if "location_city" in df.columns:
    print("Number of cities:", df["location_city"].nunique())

if all(col in df.columns for col in ["Cash_Withdrawn", "Cash_Remaining"]):
    print("\nCash_Withdrawn stats:")
    print(df["Cash_Withdrawn"].describe())
    print("\nCash_Remaining stats:")
    print(df["Cash_Remaining"].describe())

print("\nSample cleaned data:")
display(df.head(10))

# ---------- 11. PLOTS (FOR PPT) ----------

# Total cash withdrawn per day across all ATMs
if "Date" in df.columns and "Cash_Withdrawn" in df.columns:
    daily_total = df.groupby("Date")["Cash_Withdrawn"].sum()

    plt.figure(figsize=(10,4))
    daily_total.plot()
    plt.title("Total Cash Withdrawn Across All ATMs per Day")
    plt.xlabel("Date")
    plt.ylabel("Total Cash Withdrawn")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping total withdraw plot (missing Date or Cash_Withdrawn).")

# Cash remaining trend for first ATM
if "ATM_ID" in df.columns and "Cash_Remaining" in df.columns and "Date" in df.columns:
    sample_atm = df["ATM_ID"].iloc[0]
    df_atm = df[df["ATM_ID"] == sample_atm]

    plt.figure(figsize=(10,4))
    plt.plot(df_atm["Date"], df_atm["Cash_Remaining"])
    plt.title(f"Cash Remaining Over Time - {sample_atm}")
    plt.xlabel("Date")
    plt.ylabel("Cash Remaining")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping ATM cash remaining plot (missing ATM_ID, Date or Cash_Remaining).")

# ---------- 12. SAVE CLEANED DATA FOR POWER BI ----------

output_file = "DAF_PROJECT_clean_for_PowerBI.xlsx"
df.to_excel(output_file, index=False)
print("\nSaved cleaned file for Power BI as:", output_file)

# In Colab: after running, go to left panel -> Files -> right-click this file -> Download


FileNotFoundError: [Errno 2] No such file or directory: 'DAF_PROJECT.xlsx'

In [ ]:
# =======================================
# REGRESSION & CLASSIFICATION USING ALL ATMs
# =======================================

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.linear_model import LinearRegression, LogisticRegression

# We will work on a copy so we don't mess original df
data = df.copy()

# ---- 1. Prepare basic features ----

# Encode Day_of_Week as number (0-6)
if "Day_of_Week" in data.columns:
    data["Day_of_Week_Num"] = data["Day_of_Week"].astype("category").cat.codes
else:
    print("Day_of_Week not found, creating from Date")
    data["Day_of_Week"] = data["Date"].dt.day_name()
    data["Day_of_Week_Num"] = data["Day_of_Week"].astype("category").cat.codes

# Encode ATM_ID as numeric ID
data["ATM_ID_Num"] = data["ATM_ID"].astype("category").cat.codes

# Ensure key numeric columns exist
for col in ["days_since_last_refill", "balance_ratio", "Holiday_Flag"]:
    if col not in data.columns:
        print(f"WARNING: Column {col} missing, filling with 0")
        data[col] = 0

# Drop rows with missing target values (Cash_Withdrawn or refill_needed_flag)
data_reg = data.dropna(subset=["Cash_Withdrawn"])
if "refill_needed_flag" in data.columns:
    data_clf = data.dropna(subset=["refill_needed_flag"])
else:
    data_clf = None
    print("WARNING: 'refill_needed_flag' column not found for classification")

print("Total rows for regression:", len(data_reg))
if data_clf is not None:
    print("Total rows for classification:", len(data_clf))

# ------------------ REGRESSION ------------------
# Predict Cash_Withdrawn

if len(data_reg) >= 20:
    features_reg = ["ATM_ID_Num", "Day_of_Week_Num", "Holiday_Flag",
                    "days_since_last_refill", "balance_ratio"]

    X_reg = data_reg[features_reg]
    y_reg = data_reg["Cash_Withdrawn"]

    X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
        X_reg, y_reg, test_size=0.2, random_state=42
    )

    reg_model = LinearRegression()
    reg_model.fit(X_train_reg, y_train_reg)

    y_pred_reg = reg_model.predict(X_test_reg)
    mae = mean_absolute_error(y_test_reg, y_pred_reg)

    print("\n=== REGRESSION MODEL (Cash_Withdrawn) ===")
    print("Number of training samples:", len(X_train_reg))
    print("Number of test samples:", len(X_test_reg))
    print("MAE (Mean Absolute Error):", round(mae, 2))

    # Store some predictions for viewing
    data_reg.loc[X_test_reg.index, "predicted_Cash_Withdrawn"] = y_pred_reg

    print("\nSample Regression Predictions (last 10 rows):")
    display(data_reg.loc[X_test_reg.index,
                         ["ATM_ID", "Date", "Cash_Withdrawn", "predicted_Cash_Withdrawn"]].tail(10))

    # Optional: save
    data_reg.to_excel("ALL_ATM_regression_results.xlsx", index=False)
    print("Saved ALL_ATM_regression_results.xlsx")
else:
    print("\nNOT ENOUGH DATA for regression (need at least 20 rows).")

# ------------------ CLASSIFICATION ------------------
# Predict refill_needed_flag

if data_clf is not None and len(data_clf) >= 20:
    features_clf = ["ATM_ID_Num", "Day_of_Week_Num", "Holiday_Flag",
                    "days_since_last_refill", "balance_ratio"]

    X_clf = data_clf[features_clf]
    y_clf = data_clf["refill_needed_flag"].astype(int)

    X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
        X_clf, y_clf, test_size=0.2, random_state=42
    )

    clf_model = LogisticRegression(max_iter=1000)
    clf_model.fit(X_train_clf, y_train_clf)

    y_pred_clf = clf_model.predict(X_test_clf)
    acc = accuracy_score(y_test_clf, y_pred_clf)

    print("\n=== CLASSIFICATION MODEL (refill_needed_flag) ===")
    print("Number of training samples:", len(X_train_clf))
    print("Number of test samples:", len(X_test_clf))
    print("Accuracy:", round(acc * 100, 2), "%")

    data_clf.loc[X_test_clf.index, "predicted_refill_flag"] = y_pred_clf

    print("\nSample Classification Predictions (last 10 rows):")
    display(data_clf.loc[X_test_clf.index,
                         ["ATM_ID", "Date", "refill_needed_flag", "predicted_refill_flag"]].tail(10))

    data_clf.to_excel("ALL_ATM_classification_results.xlsx", index=False)
    print("Saved ALL_ATM_classification_results.xlsx")
else:
    print("\nNOT ENOUGH DATA for classification (need at least 20 rows or refill_needed_flag missing).")


Total rows for regression: 200
Total rows for classification: 200

=== REGRESSION MODEL (Cash_Withdrawn) ===
Number of training samples: 160
Number of test samples: 40
MAE (Mean Absolute Error): 631.16

Sample Regression Predictions (last 10 rows):


,ATM_ID,Date,Cash_Withdrawn,predicted_Cash_Withdrawn
9,ATM107,2024-03-18,30.0,-224.136543
18,ATM120,2023-10-13,160.0,-96.680455
55,ATM220,2023-06-18,0.0,-143.447603
75,ATM287,2023-12-29,2400.0,427.776807
150,ATM409,2024-03-22,380.0,1004.679794
104,ATM315,2023-06-15,0.0,468.632578
135,ATM401,2024-07-15,0.0,864.112274
137,ATM402,2023-11-01,3500.0,1428.315756
164,ATM422,2023-06-12,0.0,1047.672315
76,ATM289,2023-06-21,120.0,226.831833


Saved ALL_ATM_regression_results.xlsx

=== CLASSIFICATION MODEL (refill_needed_flag) ===
Number of training samples: 160
Number of test samples: 40
Accuracy: 90.0 %

Sample Classification Predictions (last 10 rows):


,ATM_ID,Date,refill_needed_flag,predicted_refill_flag
9,ATM107,2024-03-18,0,0.0
18,ATM120,2023-10-13,0,0.0
55,ATM220,2023-06-18,0,0.0
75,ATM287,2023-12-29,0,0.0
150,ATM409,2024-03-22,0,0.0
104,ATM315,2023-06-15,0,0.0
135,ATM401,2024-07-15,0,0.0
137,ATM402,2023-11-01,1,0.0
164,ATM422,2023-06-12,0,0.0
76,ATM289,2023-06-21,0,0.0


Saved ALL_ATM_classification_results.xlsx


In [ ]:
# ============================
# ATM PROJECT - FULL MODEL CODE (cleaned for sklearn)
# ============================

# 1) IMPORTS
import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# 2) UPLOAD DATA FILE FROM YOUR COMPUTER
print("Please upload DAF_PROJECT_clean_for_PowerBI.xlsx")
uploaded = files.upload()   # choose the file when the button appears

# 3) READ THE DATA
df_raw = pd.read_excel("DAF_PROJECT_clean_for_PowerBI.xlsx")

# ---- adjust these column names if yours are slightly different ----
target_class_col = 'refill_needed_flag'
target_reg_col   = 'Cash_Withdrawn'
date_col         = 'Date'
id_col           = 'ATM_ID'
# -------------------------------------------------------------------

# 4) BASIC PREPROCESSING
df = df_raw.copy()

# Convert Date to datetime and add numeric version
df[date_col] = pd.to_datetime(df[date_col])
df['Date_ordinal'] = df[date_col].map(pd.Timestamp.toordinal)

# Keep targets and IDs aside
targets_and_id = [id_col, date_col, target_class_col, target_reg_col]

# Identify feature columns (everything except id, date, targets)
feature_cols = [c for c in df.columns if c not in targets_and_id]

# From those features, drop any datetime columns (sklearn can't handle them)
non_datetime_features = [
    c for c in feature_cols
    if not np.issubdtype(df[c].dtype, np.datetime64)
]

# Separate features into X, but keep targets in df for later merging
X_full = df[non_datetime_features]

# Convert all categorical/text columns in X_full to numeric using one-hot encoding
X_dummies = pd.get_dummies(X_full, drop_first=True)

print("\nAfter cleaning + get_dummies, number of feature columns:", X_dummies.shape[1])

# ============================================================
# 5) CLASSIFICATION MODEL  (Predict refill_needed_flag)
# ============================================================

print("\n==============================")
print("=== CLASSIFICATION MODEL   ===")
print("==============================")

# Remove rows where target is missing (if any)
mask_clf = df[target_class_col].notna()
y_clf = df.loc[mask_clf, target_class_col]
X_clf = X_dummies.loc[mask_clf, :]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)
clf.fit(Xc_train, yc_train)

y_clf_pred = clf.predict(Xc_test)

# ---- Metrics ----
accuracy  = accuracy_score(yc_test, y_clf_pred)
precision = precision_score(yc_test, y_clf_pred, zero_division=0)
recall    = recall_score(yc_test, y_clf_pred, zero_division=0)
f1        = f1_score(yc_test, y_clf_pred, zero_division=0)
cm        = confusion_matrix(yc_test, y_clf_pred)

print(f"Number of training samples: {len(Xc_train)}")
print(f"Number of test samples    : {len(Xc_test)}")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print("Confusion Matrix:")
print(cm)

# ---- Save classification predictions for ALL rows ----
df_clf_results = df.copy()
df_clf_results['predicted_refill_flag'] = clf.predict(X_dummies)

print("\nSample Classification Predictions (last 10 rows):")
print(df_clf_results[[id_col, date_col, target_class_col, 'predicted_refill_flag']].tail(10))

clf_filename = "ALL_ATM_classification_results.xlsx"
df_clf_results.to_excel(clf_filename, index=False)
print(f"\nSaved {clf_filename}")

# ============================================================
# 6) REGRESSION MODEL  (Predict Cash_Withdrawn)
# ============================================================

print("\n==========================")
print("=== REGRESSION MODEL   ===")
print("==========================")

# Keep only rows where regression target is not missing
mask_reg = df[target_reg_col].notna()
y_reg = df.loc[mask_reg, target_reg_col]
X_reg = X_dummies.loc[mask_reg, :]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)
reg.fit(Xr_train, yr_train)

y_reg_pred = reg.predict(Xr_test)

# ---- Metrics ----
mae  = mean_absolute_error(yr_test, y_reg_pred)
rmse = np.sqrt(mean_squared_error(yr_test, y_reg_pred))
r2   = r2_score(yr_test, y_reg_pred)

print(f"Number of training samples: {len(Xr_train)}")
print(f"Number of test samples    : {len(Xr_test)}")
print(f"MAE  (Mean Absolute Error)     : {mae:.3f}")
print(f"RMSE (Root Mean Squared Error) : {rmse:.3f}")
print(f"R² Score                        : {r2:.4f}")

# ---- Save regression predictions for ALL rows ----
df_reg_results = df.loc[mask_reg].copy()
df_reg_results['predicted_Cash_Withdrawn'] = reg.predict(X_reg)

print("\nSample Regression Predictions (last 10 rows):")
print(df_reg_results[[id_col, date_col, target_reg_col, 'predicted_Cash_Withdrawn']].tail(10))

reg_filename = "ALL_ATM_regression_results.xlsx"
df_reg_results.to_excel(reg_filename, index=False)
print(f"\nSaved {reg_filename}")

print("\nDONE ✅")


Please upload DAF_PROJECT_clean_for_PowerBI.xlsx


Saving DAF_PROJECT_clean_for_PowerBI.xlsx to DAF_PROJECT_clean_for_PowerBI (1).xlsx

After cleaning + get_dummies, number of feature columns: 849

=== CLASSIFICATION MODEL   ===
Number of training samples: 160
Number of test samples    : 40
Accuracy  : 0.9250
Precision : 0.8333
Recall    : 0.7143
F1 Score  : 0.7692
Confusion Matrix:
[[32  1]
 [ 2  5]]

Sample Classification Predictions (last 10 rows):
     ATM_ID       Date  refill_needed_flag  predicted_refill_flag
190  ATM950 2023-11-01                   0                      0
191  ATM950 2023-11-01                   0                      0
192  ATM950 2023-11-02                   0                      0
193  ATM950 2023-11-02                   0                      0
194  ATM950 2023-11-02                   0                      0
195  ATM950 2023-12-31                   1                      1
196  ATM970 2024-03-01                   0                      0
197  ATM970 2024-03-01                   0                      0
1

In [ ]:
# ---------------------------------------------------
# STEP 1: Import libraries
# ---------------------------------------------------
import pandas as pd
from google.colab import files
import io

print("Upload your dataset (CSV or Excel)")
uploaded = files.upload()

# Get the uploaded file name
file_name = list(uploaded.keys())[0]
print("File uploaded:", file_name)

# Convert uploaded content into bytes
data = io.BytesIO(uploaded[file_name])

# ---------------------------------------------------
# STEP 2: Try reading Excel first, then CSV automatically
# ---------------------------------------------------
df = None

try:
    print("\nTrying to read as Excel (.xlsx)...")
    df = pd.read_excel(data)
    print("Loaded as Excel file successfully ✔")
except:
    print("Excel load failed. Trying CSV...")
    try:
        data.seek(0)
        df = pd.read_csv(data, encoding='latin1')  # latin1 avoids utf-8 errors
        print("Loaded as CSV successfully ✔")
    except Exception as e:
        print("\n❌ Could NOT read file.")
        print("Error:", e)
        raise SystemExit("Please re-upload the file in Excel (.xlsx) format.")

# ---------------------------------------------------
# STEP 3: Basic Cleaning
# ---------------------------------------------------
df = df.drop_duplicates()
df = df.fillna(0)

# ---------------------------------------------------
# STEP 4: Preview (for checking)
# ---------------------------------------------------
print("\nTop records (df.head()):")
display(df.head())

print("\nDataset Info (df.info()):")
buffer = io.StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

# ---------------------------------------------------
# STEP 5: PARTIAL OUTPUT (USE THIS IN SLIDE 6)
# ---------------------------------------------------
print("\nSummary Statistics (df.describe()):")
display(df.describe())

Upload your dataset (CSV or Excel)


Saving DAF_PROJECT.csv to DAF_PROJECT (1).csv
File uploaded: DAF_PROJECT (1).csv

Trying to read as Excel (.xlsx)...
Excel load failed. Trying CSV...
Loaded as CSV successfully ✔

Top records (df.head()):


,ATM_ID,Branch,location_city,location_street_address,location_postal_code,location_country,Max_Cash_Capacity,Date,Day_of_Week,Holiday_Flag,Withdrawals_Count,Cash_Withdrawn,Last_Refill_Date,Cash_Refilled,Cash_Remaining,transaction_date,transaction_time,transaction_type,withdrawal_amount,refill_amount,atm_balance_after_transaction,currency,refill_operator_id,avg_daily_withdrawal,demand_tier,last_refill_date,days_since_last_refill,balance_ratio,refill_needed_flag,suggested_refill_amount
0,ATM101,Main Street Branch,San Francisco,55 Main St,94105,USA,13040.0,2023-12-14 00:00:00,Thursday,0,1,40.0,0,0,9820.0,2023-12-14 00:00:00,08:12:44,withdrawal,40.0,0,9820.0,USD,0,80.0,Tier 2,0,0.0,0.753067,0,0.0
1,ATM101,Downtown Branch,New York,123 Main St,10001,USA,13040.0,2024-03-15 00:00:00,Friday,0,1,120.0,0,0,13040.0,2024-03-15 00:00:00,08:15:22,withdrawal,120.0,0,13040.0,USD,0,80.0,Tier 2,0,0.0,1.000000,0,0.0
2,ATM102,Grand Plaza Mall,Los Angeles,456 Market Ave,90017,USA,15420.0,2024-03-15 00:00:00,Friday,0,1,60.0,0,0,15420.0,2024-03-15 00:00:00,12:45:09,withdrawal,60.0,0,15420.0,USD,0,60.0,Tier 2,0,0.0,1.000000,0,0.0
3,ATM103,Main St Branch,Houston,100 Main St,77002,USA,13000.0,2023-06-20 00:00:00,Tuesday,0,1,200.0,0,0,13000.0,2023-06-20 00:00:00,12:00:11,withdrawal,200.0,0,13000.0,USD,0,220.0,Tier 2,0,0.0,1.000000,0,0.0
4,ATM103,Eastside Grocery,Chicago,789 Broadway,60601,USA,13000.0,2024-03-16 00:00:00,Saturday,1,1,240.0,0,0,11260.0,2024-03-16 00:00:00,18:32:41,withdrawal,240.0,0,11260.0,USD,0,220.0,Tier 2,0,0.0,0.866154,0,0.0



Dataset Info (df.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 30 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   ATM_ID                         200 non-null    object 
 1   Branch                         200 non-null    object 
 2   location_city                  200 non-null    object 
 3   location_street_address        200 non-null    object 
 4   location_postal_code           200 non-null    object 
 5   location_country               200 non-null    object 
 6   Max_Cash_Capacity              200 non-null    float64
 7   Date                           200 non-null    object 
 8   Day_of_Week                    200 non-null    object 
 9   Holiday_Flag                   200 non-null    int64  
 10  Withdrawals_Count              200 non-null    int64  
 11  Cash_Withdrawn                 200 non-null    float64
 12  Last_Refill_Date       

,Max_Cash_Capacity,Holiday_Flag,Withdrawals_Count,Cash_Withdrawn,Cash_Refilled,Cash_Remaining,withdrawal_amount,refill_amount,atm_balance_after_transaction,avg_daily_withdrawal,days_since_last_refill,balance_ratio,refill_needed_flag,suggested_refill_amount
count,2.000000e+02,200.000000,200.000000,200.000000,2.000000e+02,2.000000e+02,200.000000,2.000000e+02,2.000000e+02,200.000000,200.000000,200.000000,200.000000,200.000000
mean,6.337866e+04,0.295000,0.860000,548.922700,3.960752e+04,5.174751e+04,382.912650,3.668002e+04,5.174751e+04,641.555192,6.310000,0.788408,0.145000,7898.459750
std,2.252512e+05,0.457187,0.702043,1392.238103,1.961054e+05,2.208789e+05,1194.859309,1.963493e+05,2.208789e+05,1307.740497,30.648828,0.331230,0.352984,39799.141775
min,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.015000e+03,0.000000,0.000000,0.000000,0.000000e+00,3.650000e+03,0.000000,0.000000e+00,3.650000e+03,15.000000,0.000000,0.591294,0.000000,0.000000
50%,1.230000e+04,0.000000,1.000000,60.000000,0.000000e+00,9.730000e+03,37.500000,0.000000e+00,9.730000e+03,86.670000,0.000000,1.000000,0.000000,0.000000
75%,2.125000e+04,1.000000,1.000000,300.250000,1.700000e+04,1.912975e+04,182.500000,1.000000e+04,1.912975e+04,380.000000,0.000000,1.000000,0.000000,0.000000
max,1.500000e+06,1.000000,3.000000,10000.000000,1.500000e+06,1.500000e+06,10000.000000,1.500000e+06,1.500000e+06,10000.000000,212.000000,1.000000,1.000000,294000.000000


...

from matplotlib import pyplot as plt
_df_7.plot(kind='scatter', x='Cash_Withdrawn', y='Cash_Refilled', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_8['Max_Cash_Capacity'].plot(kind='line', figsize=(8, 4), title='Max_Cash_Capacity')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_9['Holiday_Flag'].plot(kind='line', figsize=(8, 4), title='Holiday_Flag')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_10['Withdrawals_Count'].plot(kind='line', figsize=(8, 4), title='Withdrawals_Count')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_11['Cash_Withdrawn'].plot(kind='line', figsize=(8, 4), title='Cash_Withdrawn')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
# 1️⃣ Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 2️⃣ Load your cleaned dataset
# Replace 'cleaned_dataset.csv' with your file
df = pd.read_csv('cleaned_dataset.csv')

# 3️⃣ Missing Value Analysis
missing_values = df.isnull().sum()
print("Missing Values After Cleaning:\n")
print(missing_values)

# Optional: heatmap for missing values (visual)
plt.figure(figsize=(8,4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title("Missing Values Heatmap After Cleaning")
plt.show()

# 4️⃣ Simple Exploratory Graphs

# Example 1: Histogram of a numeric column
plt.figure(figsize=(6,4))
plt.hist(df['Maths'], bins=10, color='skyblue', edgecolor='black')
plt.title('Distribution of Maths Scores')
plt.xlabel('Maths')
plt.ylabel('Frequency')
plt.show()

# Example 2: Boxplot of another numeric column
plt.figure(figsize=(6,4))
plt.boxplot(df['Annual'])
plt.title('Boxplot of Annual Column')
plt.show()

# Example 3: Bar plot of a categorical column
plt.figure(figsize=(6,4))
df['Holiday_Flag'].value_counts().plot(kind='bar', color='orange')
plt.title('Holiday Flag Count')
plt.xlabel('Holiday Flag')
plt.ylabel('Count')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_dataset.csv'